In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Data loading
df = pd.read_csv("netflix.csv")

print("=" * 50)
print("Netflix analysis")
print("=" * 50)

print(df.head())
print(df.tail())

print("\nDataset shape:")
print(df.shape)

print("\nDataset information:")
print(df.info())

In [ ]:
# DATA CLEANING

df['Show_ID'] = df['Show_ID'].fillna(df["Show_ID"].mode()[0])
print("\nMissing values in Show_ID after imputation")

df['Type'] = df['Type'].fillna(df["Type"].mode()[0])
print("\nMissing values in Type after imputation")

df['Title'] = df['Title'].fillna(df["Type"].mode()[0])
print("\nMissing values in Title after imputation")

df['Director'] = df['Director'].fillna("unknown")
print("\nMissing values in Director after imputation")

df['Cast'] = df['Cast'].fillna(df["Cast"].mode()[0])
print("\nMissing values in Cast after imputation")

df['Country'] = df['Country'].fillna(df["Country"].mode()[0])
print("\nMissing values in Country after imputation")

df['Release_Year'] = df['Release_Year'].fillna(df["Release_Year"].mean())
print("\nMissing values in Release_Year after imputation")

df['Rating'] = df['Rating'].fillna(df["Rating"].mode()[0])
print("\nMissing values in Rating after imputation")

df['Duration'] = df['Duration'].fillna(df["Duration"].mode()[0])
print("\nMissing values in Duration after imputation")

df['Description'] = df['Description'].fillna(df["Description"].mode()[0])
print("\nMissing values in Description after imputation")

df = df.drop_duplicates()
print("\nDataset shape after removing duplicates:")
print(df.shape)

df["Director"] = df["Director"].str.title()
df["Country"] = df["Country"].str.strip()
df["Type"] = df["Type"].str.title()

df["Duration"] = df["Duration"].str.replace("min", " min", regex=False)
df["Duration"] = df["Duration"].str.replace(" ", " ")

df["Country"] = df["Country"].str.title()

df["Date_Added"] = df["Date_Added"].str.strip()
df["Date_Added"] = pd.to_datetime(df["Date_Added"], errors="coerce")
df["Date_Added"] = df["Date_Added"].dt.strftime("%Y-%m-%d")
df["Date_Added"] = df["Date_Added"].fillna("unknown")

print("\nMissing values after cleaning:")
print(df.isnull().sum())

df.to_csv("output/cleaned netflix.csv", index=False)

In [ ]:
# Exploratory Data Analysis

# Total number of Netflix titles
print("Total number of Netflix titles:", len(df))

# Total number of TV shows and movies
print("\nTotal number of TV shows and movies:")
print(df["Type"].value_counts())

# Oldest and newest release year
print("\nOldest release year:", df["Release_Year"].min())
print("Newest release year:", df["Release_Year"].max())

# Average release year
print("\nAverage release year:", df["Release_Year"].mean())

# Count content by rating
print("\nCount content by rating:")
print(df["Rating"].value_counts())

# Top 10 most common genres
top_10 = df["Genre (Listed_In)"].value_counts().head(10)
print("\nTop 10 genres:")
print(top_10)

# Top 10 countries with the most Netflix content
top_10 = df["Country"].value_counts().head(10)
print("\nTop 10 countries:")
print(top_10)

# Top 10 directors with the highest number of titles
top_10 = df["Director"].value_counts().head(10)
print("\nTop 10 directors:")
print(top_10)

# Year with the highest number of releases
yearly_release_counts = df["Release_Year"].value_counts()
print("\nYear with highest releases:", yearly_release_counts.idxmax())
print("Number of releases:", yearly_release_counts.max())

# Month in which Netflix added the most content
df["Date_Added"] = df["Date_Added"].replace("Unknown", pd.NA)
df["Date_Added"] = pd.to_datetime(df["Date_Added"], errors="coerce")

month_count = df["Date_Added"].dt.month_name().value_counts()
print("\nMonth with most added content:", month_count.idxmax())
print("Number of titles:", month_count.max())

# Movies released after 2020
movies_after_2020 = df[(df["Type"] == "Movie") & (df["Release_Year"] > 2020)]
print("\nNumber of movies released after 2020:", len(movies_after_2020))

# TV Shows with more than 3 seasons
# Note: this assumes TV Show Duration values are stored as season counts.
tv_show_duration = pd.to_numeric(
    df.loc[df["Type"] == "Tv Show", "Duration"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)
tv_shows_with_more_than_3_seasons = df[
    (df["Type"] == "Tv Show") &
    (pd.to_numeric(df["Duration"].astype(str).str.extract(r"(\d+)")[0], errors="coerce") > 3)
]
print("Number of TV shows with more than 3 seasons:", len(tv_shows_with_more_than_3_seasons))

# Content released in India
Indian_content = df[df["Country"] == "India"]
print("\nNumber of content released in India:", len(Indian_content))

# Content directed by a specific director
director_content = df[df["Director"] == "Specific Director"]
print("Number of content directed by Specific Director:", len(director_content))

# Titles containing the word Love
love_titles = df[df["Title"].astype(str).str.contains("Love", case=False, na=False)]
print("Number of titles containing Love:", len(love_titles))

# Count Movies and TV Shows year-wise
yearly_counts = df.groupby(["Release_Year", "Type"]).size()
print("\nYear-wise count of Movies and TV Shows:")
print(yearly_counts)

# Most common content rating
print("\nMost common content rating:", df["Rating"].mode()[0])

# Longest and shortest movie
# Extract numeric duration so comparison works correctly.
movie_duration_num = pd.to_numeric(
    df.loc[df["Type"] == "Movie", "Duration"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

movie_rows = df[df["Type"] == "Movie"].copy()
movie_rows["Duration_Num"] = movie_duration_num

if not movie_rows.empty and movie_rows["Duration_Num"].notna().any():
    longest_movie = movie_rows.loc[movie_rows["Duration_Num"].idxmax()]
    shortest_movie = movie_rows.loc[movie_rows["Duration_Num"].idxmin()]
    print("Longest movie is:", longest_movie["Title"], "-", longest_movie["Duration"])
    print("Shortest movie is:", shortest_movie["Title"], "-", shortest_movie["Duration"])

# Top 10 latest releases by release year
top_10_latest_releases = df.sort_values("Release_Year", ascending=False).head(10)
print("\nTop 10 latest releases:")
print(top_10_latest_releases["Title"].tolist())

# Oldest 10 titles by release year
oldest_10_titles = df.sort_values("Release_Year", ascending=True).head(10)
print("\nTop 10 oldest titles:")
print(oldest_10_titles["Title"].tolist())

# Genre-wise content count
genre_counts = df["Genre (Listed_In)"].value_counts()
print("\nGenre-wise content count:")
print(genre_counts)

# Country-wise average release year
country_avg_release_year = df.groupby("Country")["Release_Year"].mean()
print("\nCountry-wise average release year:")
print(country_avg_release_year)

In [ ]:
# VISUALIZATION

# Create a Pie Chart showing Movies vs TV Shows
type_counts = df["Type"].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%')
plt.title('Distribution of Movies vs TV Shows')
plt.show()

In [ ]:
# Create a Bar Chart for the Top 10 Genres
top_10_genres = df["Genre (Listed_In)"].value_counts().head(10)

plt.figure(figsize=(10, 6))
plt.bar(top_10_genres.index, top_10_genres.values)
plt.title('Top 10 Genres')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Create a Bar Chart for the Top 10 Countries
top_10_countries = df["Country"].value_counts().head(10)

plt.figure(figsize=(10, 6))
plt.bar(top_10_countries.index, top_10_countries.values)
plt.title('Top 10 Countries')
plt.xlabel('Country')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Create a Histogram for Release Year Distribution
plt.figure(figsize=(10, 6))
plt.hist(df["Release_Year"], bins=20, edgecolor='black')
plt.title("Release Year Distribution")
plt.xlabel("Release Year")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create a Count Plot for Content Ratings
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="Rating", order=df["Rating"].value_counts().index)
plt.title("Content Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Create a Horizontal Bar Chart for the Top 10 Directors
top_10_directors = df["Director"].value_counts().head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_10_directors.index, top_10_directors.values)
plt.title("Top 10 Directors")
plt.xlabel("Count")
plt.ylabel("Director")
plt.show()

In [ ]:
# Create a Line Chart showing Releases by Year
plt.figure(figsize=(10, 6))
yearly_counts = df.groupby("Release_Year").size()

plt.plot(yearly_counts.index, yearly_counts.values)
plt.title("Releases by Year")
plt.xlabel("Release Year")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create a Box Plot for Movie Duration
movie_duration = pd.to_numeric(
    df.loc[df["Type"] == "Movie", "Duration"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
).dropna()

plt.figure(figsize=(10, 6))
sns.boxplot(x=movie_duration)
plt.title("Distribution of Movie Duration")
plt.xlabel("Duration (minutes)")
plt.show()

In [ ]:
# Create a Heatmap of correlations between numerical features
plot_data = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 6))
sns.heatmap(plot_data.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap between Numerical Features")
plt.show()

In [ ]:
# Create a Dashboard containing at least six visualizations

plt.figure(figsize=(18, 10))

# 1. Movies vs TV Shows
plt.subplot(2, 3, 1)
df["Type"].value_counts().plot(kind="bar")
plt.title("Movies vs TV Shows")
plt.xlabel("Type")
plt.ylabel("Count")

# 2. Top 5 Countries
plt.subplot(2, 3, 2)
df["Country"].value_counts().head(5).plot(kind="bar")
plt.title("Top 5 Countries")
plt.xlabel("Country")
plt.ylabel("Count")

# 3. Top 5 Genres
plt.subplot(2, 3, 3)
df["Genre (Listed_In)"].value_counts().head(5).plot(kind="bar")
plt.title("Top 5 Genres")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")

# 4. Content Ratings
plt.subplot(2, 3, 4)
df["Rating"].value_counts().plot(kind="bar")
plt.title("Content Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.xticks(rotation=45)

# 5. Releases by Year
plt.subplot(2, 3, 5)
df["Release_Year"].value_counts().sort_index().plot(kind="line", marker="o")
plt.title("Releases by Year")
plt.xlabel("Release Year")
plt.ylabel("Count")

# 6. Top 10 Directors
plt.subplot(2, 3, 6)
df["Director"].value_counts().head(10).plot(kind="barh")
plt.title("Top 10 Directors")
plt.xlabel("Count")
plt.ylabel("Director")

plt.tight_layout(pad=3)
plt.show()